# 2 · Claude Code I: AI as a Coding Copilot

**The question of this session:** the market prices Apple at **$309.35 a share** — **$4.51 trillion** in total. Is that justified? You will answer the way an analyst does: value Apple against its nine largest technology peers, using their own filings with the U.S. Securities and Exchange Commission (SEC), and publish the tool that does it to your GitHub.

**In this notebook you will:**

- Fetch the peers' fundamentals live from their SEC filings
- Compute what the market charges per unit of earnings, across the peer group
- Value Apple at its peers' ratings, end to end — as a point, then as a range
- Publish your first finance project to GitHub

## The working rhythm
With Claude Code you are the analyst in charge; the model is a fast assistant:

1. **Ask small.** One function, one fix, one chart at a time.
2. **Read before you run.** Accept only changes you can explain.
3. **Commit at every green moment.** Small commits make each step reversible.

**How to use Claude Code in this notebook:** select an exercise's docstring, press `Option+K` (Mac) / `Alt+K` (Windows): the file and lines land in the ✱ panel: then ask *"implement this"*. Read the diff it proposes before accepting.

**Pandas in one paragraph:** a `DataFrame` is the analyst's table. `pd.read_csv` loads it; columns are vectors, so `df["a"] / df["b"]` computes a whole ratio column at once; you don't memorize pandas: you *specify* what you want and *verify* what you get.

In [1]:
import sys, os, json
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
# If you pulled course updates while this kernel was running, pick them up here
# (a no-op on a fresh kernel; saves a restart otherwise):
import importlib
for _n in [n for n in list(sys.modules) if n.startswith("toolkit")]:
    importlib.reload(sys.modules[_n])
try:
    from dotenv import load_dotenv
    load_dotenv(ROOT / ".env")
except ImportError:
    pass
HAS_KEY = bool(os.environ.get("ANTHROPIC_API_KEY"))
print(f"repo root: {ROOT}")
print(f"API key:   {'configured' if HAS_KEY else 'NOT SET - cells that call Claude will be skipped'}")

repo root: /Users/ignaciocastro/iese-ai-finance-bootcamp-student
API key:   configured


## Part A · The question, and the data

$309.35 is a **price**: it emerged from Friday's trading. Today we compute an independent estimate of Apple's **worth**, by asking what the prices of its peers imply. The method is **comparable-company analysis**.

The data: **Apple and seven peers** — Microsoft, Alphabet, Meta, Amazon, NVIDIA, Oracle and Salesforce. The fundamentals are fetched **live from SEC EDGAR** when you run the cell below. EDGAR is the U.S. Securities and Exchange Commission's public filing database: every listed company is legally required to file its official reports there (the annual 10-K among them), and EDGAR serves them free and machine-readable — so these are the audited numbers each company signed, straight from the source (results are cached locally; if the connection fails, a bundled copy loads instead). Prices are from Friday's close. Four definitions carry the session:

- **EBITDA** (earnings before interest, taxes, depreciation and amortization) **≈ operating income + depreciation & amortization (D&A)** — what the operations earn, before financing and accounting choices.
- **Market capitalization = shares × share price** — the price of the equity.
- **Enterprise value (EV) = market cap + debt − cash** — the price of the whole business.
- **EV/EBITDA = enterprise value ÷ EBITDA** — the whole business per unit of what it earns. **A multiple is a rating, not a price**: it becomes an estimate of worth only when a *peer's* rating is applied to Apple's own earnings — which is exactly what Part B does.

Load the dataset:

In [2]:
import numpy as np
import pandas as pd
import csv
pd.options.display.float_format = "{:,.1f}".format

TICKERS = ["AAPL", "MSFT", "NVDA", "GOOGL", "AMZN", "META", "CRM", "ORCL"]
DATA = ROOT / "session-02-coding-copilot" / "data"

try:
    # Live: for each company, toolkit/edgar.py requests its filing facts from
    #   https://data.sec.gov/api/xbrl/companyfacts/CIK##########.json
    # The SEC requires each request to identify its sender - that is the
    # SEC_EDGAR_USER_AGENT line in your .env. Responses are cached in .cache/
    # for 24 hours, so only the first run actually crosses the network.
    sys.path.insert(0, str(DATA))
    from make_dataset import build_row
    with open(DATA / "prices.csv", newline="") as f:
        prices = {r["ticker"]: {"price_usd": float(r["price_usd"]), "price_asof": r["price_asof"]}
                  for r in csv.DictReader(f)}
    print("Loading data live from SEC EDGAR (data.sec.gov) ...")
    rows = []
    for t in TICKERS:
        print(f"  {t:5} latest 10-K facts ... fetched")
        rows.append(build_row(t, prices))
    companies = pd.DataFrame(rows)
    print("done.")
    source = "SEC EDGAR, live (cached locally for 24h after the first run)"
except Exception as exc:
    # Offline fallback: the same data, bundled with the course.
    companies = pd.read_csv(DATA / "tech_financials.csv")
    companies = companies[companies["ticker"].isin(TICKERS)].reset_index(drop=True)
    source = f"bundled CSV (live fetch unavailable: {type(exc).__name__})"

print(f"\n{len(companies)} companies | fundamentals: {source} | prices as of {companies['price_asof'].iloc[0]}")
companies

Loading data live from SEC EDGAR (data.sec.gov) ...
  AAPL  latest 10-K facts ... fetched
  MSFT  latest 10-K facts ... fetched
  NVDA  latest 10-K facts ... fetched
  GOOGL latest 10-K facts ... fetched
  AMZN  latest 10-K facts ... fetched
  META  latest 10-K facts ... fetched
  CRM   latest 10-K facts ... fetched
  ORCL  latest 10-K facts ... fetched
done.

8 companies | fundamentals: SEC EDGAR, live (cached locally for 24h after the first run) | prices as of 2026-08-21


,ticker,company,fy_end,revenue_m,revenue_prior_m,revenue_prior2_m,operating_income_m,net_income_m,d_and_a_m,cash_m,total_debt_m,shares_m,price_usd,price_asof
0,AAPL,Apple Inc.,2025-09-27,"416,161.0","391,035.0","383,285.0","133,050.0","112,010.0","11,698.0","39,544.0","84,344.0","14,594.2",309.4,2026-08-21
1,MSFT,MICROSOFT CORP,2026-06-30,"331,839.0","281,724.0","245,122.0","155,237.0","133,749.0","34,300.0","20,935.0","40,294.0","7,425.5",483.2,2026-08-21
2,NVDA,NVIDIA CORP,2026-01-25,"215,938.0","130,497.0","60,922.0","130,387.0","120,067.0","2,843.0","13,237.0","8,470.0","24,200.0",214.7,2026-08-21
3,GOOGL,Alphabet Inc.,2025-12-31,"402,836.0","350,018.0","307,394.0","129,039.0","132,170.0","21,136.0","55,911.0","100,164.0","12,230.0",344.8,2026-08-21
4,AMZN,AMAZON COM INC,2025-12-31,"716,924.0","637,959.0","574,785.0","79,975.0","77,670.0","65,756.0","78,213.0","132,995.0","10,786.3",258.6,2026-08-21
5,META,"Meta Platforms, Inc.",2025-12-31,"200,966.0","164,501.0","134,902.0","83,276.0","60,458.0","18,616.0","15,462.0","83,664.0","2,574.0",549.9,2026-08-21
6,CRM,"Salesforce, Inc.",2026-01-31,"41,525.0","37,895.0","34,857.0","8,331.0","7,457.0","1,200.0","8,935.0","39,280.0",819.0,209.2,2026-08-21
7,ORCL,ORACLE CORP,2026-05-31,"67,357.0","57,399.0","52,961.0","20,606.0","17,087.0","7,623.0","31,289.0","129,541.0","2,880.5",146.5,2026-08-21


### Exercise 1: know the peer group

Before pricing Apple off its peers, know who the peers are: how fast each grows, and what margin it earns. One term the exercise uses: the **compound annual growth rate (CAGR)** is the single constant yearly growth rate that would carry the first year's revenue to the last year's — a fair way to compare growth across companies over the same period. These two columns are also what you will hover on in the chart, and what Session 4's screening engine filters on.

Add five columns: `revenue_growth_1y` (vs prior), `revenue_cagr_2y` (two-year compound growth: `(rev/rev_prior2)**0.5 - 1`), `ebitda_m`, `op_margin`, `ebitda_margin`.

In [3]:
def add_growth_and_margins(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
### START CODE HERE ###
    df["revenue_growth_1y"] = df["revenue_m"] / df["revenue_prior_m"] - 1
    df["revenue_cagr_2y"] = (df["revenue_m"] / df["revenue_prior2_m"]) ** 0.5 - 1    # exponent for a TWO-year CAGR?
    df["ebitda_m"] = df["operating_income_m"] + df["d_and_a_m"]                         # EBITDA = which two columns?
    df["op_margin"] = df["operating_income_m"] / df["revenue_m"]
    df["ebitda_margin"] = df["ebitda_m"] / df["revenue_m"]
### END CODE HERE ###
    return df

companies = add_growth_and_margins(companies)
companies[["ticker", "revenue_growth_1y", "ebitda_margin"]].round(3)

,ticker,revenue_growth_1y,ebitda_margin
0,AAPL,0.1,0.3
1,MSFT,0.2,0.6
2,NVDA,0.7,0.6
3,GOOGL,0.2,0.4
4,AMZN,0.1,0.2
5,META,0.2,0.5
6,CRM,0.1,0.2
7,ORCL,0.2,0.4


In [4]:
# ✅ self-check: run me
r = companies.set_index("ticker")
assert "revenue_growth_1y" in companies and "ebitda_margin" in companies, "missing columns"
assert abs(r.loc["AAPL", "revenue_growth_1y"] - (r.loc["AAPL", "revenue_m"] / r.loc["AAPL", "revenue_prior_m"] - 1)) < 1e-9
assert ((companies["op_margin"] > -1) & (companies["op_margin"] < 1)).all(), "a margin outside (-100%, 100%) means a unit error"
assert (companies["revenue_cagr_2y"].abs() < 1.5).all(), "CAGR out of range - check the exponent (2-year: **0.5)"
print("All checks passed ✅")

All checks passed ✅


### Exercise 2: what the market charges per unit of earnings

For each company: build the enterprise-value bridge, then three multiples — **EV/EBITDA** (the anchor), **EV/Sales** (EV ÷ revenue), and **price-to-earnings (P/E)** (market cap ÷ net income, the equity per unit of net profit).

One convention to build in from the start: **a multiple over a negative denominator is meaningless**. Your code must return `NaN` in that case — reported as **n.m.** (not meaningful) — never a negative multiple. Today's eight companies are all profitable, so the guard will not fire here; the screens of Session 4 meet companies where it does.

In [5]:
def add_multiples(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
### START CODE HERE ###
    df["mcap_m"] = df["shares_m"] * df["price_usd"]                            # market cap = shares x ?
    df["ev_m"] = df["mcap_m"] + df["total_debt_m"] - df["cash_m"]              # EV = mcap + debt - cash
    df["ev_ebitda"] = np.where(df["ebitda_m"] > 0, df["ev_m"] / df["ebitda_m"], np.nan)
    df["ev_sales"] = df["ev_m"] / df["revenue_m"]
    df["pe"] = np.where(df["net_income_m"] > 0, df["mcap_m"] / df["net_income_m"], np.nan)
### END CODE HERE ###
    return df

companies = add_multiples(companies)
companies[["ticker", "ev_ebitda", "ev_sales", "pe"]].round(1)

,ticker,ev_ebitda,ev_sales,pe
0,AAPL,31.5,11.0,40.3
1,MSFT,19.0,10.9,26.8
2,NVDA,39.0,24.0,43.3
3,GOOGL,28.4,10.6,31.9
4,AMZN,19.5,4.0,35.9
5,META,14.6,7.4,23.4
6,CRM,21.2,4.9,23.0
7,ORCL,18.4,7.7,24.7


In [6]:
# ✅ self-check: run me
r = companies.set_index("ticker")
assert 5 < r.loc["AAPL", "ev_ebitda"] < 60, "AAPL EV/EBITDA looks wrong - check EV = mcap + debt - cash"
assert (companies["ev_ebitda"].dropna() > 0).all(), "no negative multiples allowed"
assert {"mcap_m", "ev_m", "ev_ebitda", "ev_sales", "pe"} <= set(companies.columns), "a multiple column is missing"
neg = pd.DataFrame({"ticker": ["LOSSCO"], "mcap_m": [100.0], "net_income_m": [-5.0]})
assert np.isnan(np.where(neg["net_income_m"] > 0, neg["mcap_m"] / neg["net_income_m"], np.nan)).all(), \
    "the n.m. guard: a negative denominator must yield NaN"
print("All checks passed ✅")

All checks passed ✅


### Exercise 3: the summary an analyst could actually use

One table, sorted by EV/EBITDA, with a **MEDIAN** row at the bottom. That median is the anchor: it is the multiple the deterministic valuation of Apple applies in Part B.

Return `ticker, revenue_m, revenue_growth_1y, ebitda_margin, ev_ebitda, ev_sales, pe`, sorted by `ev_ebitda` ascending (NaN last), plus a final `MEDIAN` row with column medians.

In [7]:
def build_summary(df: pd.DataFrame) -> pd.DataFrame:
    cols = ["ticker", "revenue_m", "revenue_growth_1y", "ebitda_margin", "ev_ebitda", "ev_sales", "pe"]
### START CODE HERE ###
    summary = df[cols].sort_values("ev_ebitda", na_position="last")      # sort by which multiple?
    median = summary.drop(columns="ticker").median(numeric_only=True)
    summary = pd.concat([summary, pd.DataFrame([{"ticker": "MEDIAN", **median.to_dict()}])],
                        ignore_index=True)                        # label for the final row?
### END CODE HERE ###
    return summary

summary = build_summary(companies)
summary.round(2)

,ticker,revenue_m,revenue_growth_1y,ebitda_margin,ev_ebitda,ev_sales,pe
0,META,"200,966.0",0.2,0.5,14.6,7.4,23.4
1,ORCL,"67,357.0",0.2,0.4,18.4,7.7,24.7
2,MSFT,"331,839.0",0.2,0.6,19.0,10.9,26.8
3,AMZN,"716,924.0",0.1,0.2,19.5,4.0,35.9
4,CRM,"41,525.0",0.1,0.2,21.2,4.9,23.0
5,GOOGL,"402,836.0",0.1,0.4,28.4,10.6,31.9
6,AAPL,"416,161.0",0.1,0.3,31.5,11.0,40.3
7,NVDA,"215,938.0",0.7,0.6,39.0,24.0,43.3
8,MEDIAN,"273,888.5",0.2,0.4,20.3,9.2,29.4


In [8]:
# ✅ self-check: run me
assert summary.iloc[-1]["ticker"] == "MEDIAN", "last row must be the MEDIAN"
assert len(summary) == len(companies) + 1
v = summary["ev_ebitda"].dropna().iloc[:-1]
assert (v.values == sorted(v.values)).all(), "sort by ev_ebitda ascending"
print("All checks passed ✅")

OUTD = ROOT / "outputs"; OUTD.mkdir(exist_ok=True)
summary.to_csv(OUTD / "comps_summary.csv", index=False)
print("Saved outputs/comps_summary.csv - this file goes in your portfolio.")

All checks passed ✅
Saved outputs/comps_summary.csv - this file goes in your portfolio.


### The chart: multiples at a glance

A table answers precise questions; a chart shows who stands out. This one is **interactive**: hover a bar for the company's growth and margin, drag to zoom, double-click to reset. (Plotly draws interactive charts; seaborn and matplotlib, which the course also installs, draw static ones.)

In [9]:
# plotly draws the chart; its notebook renderer needs nbformat. Install either
# if missing (happens on environments created before they joined the course).
import importlib, subprocess, sys
for _pkg in ("plotly", "nbformat"):
    try:
        importlib.import_module(_pkg)
    except ModuleNotFoundError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", _pkg], check=True)
import plotly.express as px

plot_df = companies.dropna(subset=["ev_ebitda"]).sort_values("ev_ebitda")
fig = px.bar(
    plot_df, x="ticker", y="ev_ebitda",
    hover_data={"revenue_growth_1y": ":.1%", "ebitda_margin": ":.1%", "ev_ebitda": ":.1f"},
    labels={"ev_ebitda": "EV/EBITDA (x)", "ticker": ""},
    title="EV/EBITDA by company (a company with negative EBITDA would show n.m. and be excluded)",
)
fig.add_hline(y=plot_df["ev_ebitda"].median(), line_dash="dash", annotation_text="median")
fig.show()

## Part B · Value Apple

**What we compare, exactly.** Share prices cannot be compared across companies: price per share depends on how many shares a company happens to have (a 2-for-1 stock split halves the price and changes nothing about the business). What *is* comparable is a **rating** — price per dollar of earnings — because the arbitrary share count cancels out. The real comparison of this session is therefore Apple's rating against its peers' ratings. The chain below only converts the peers' rating back into dollars per Apple share, so the verdict can be read against the one number everyone knows: the actual market price.

The chain, from a peer rating to a share price:

```
1. peer EV/EBITDA rating × Apple's EBITDA       =  implied enterprise value
2. implied EV − debt + cash                     =  implied equity value
3. implied equity value ÷ shares                =  implied share price
```

Step 2 is your own Exercise 2 bridge, walked backwards. The peers are simply **the other seven companies**. The cell below runs the chain once, the **deterministic** way — the *median* of the seven ratings — printing every intermediate value. Read it line by line: this is a company being valued, end to end.

In [10]:
# The deterministic version: one rating (the peer median), one price. GIVEN - read it.
target = companies.set_index("ticker").loc["AAPL"]
peer_ratings = companies.loc[companies["ticker"] != "AAPL"].set_index("ticker")["ev_ebitda"]
print("The seven peer ratings (EV/EBITDA):")
print(peer_ratings.round(1).to_string(), "\n")

median_rating  = peer_ratings.median()
implied_ev     = median_rating * target["ebitda_m"]                         # step 1
implied_equity = implied_ev - target["total_debt_m"] + target["cash_m"]     # step 2: bridge, backwards
implied_price  = implied_equity / target["shares_m"]                        # step 3

print(f"peer median EV/EBITDA        : {median_rating:8.1f}x")
print(f"x Apple's EBITDA             : {target['ebitda_m']:8,.0f}m")
print(f"= implied enterprise value   : {implied_ev:8,.0f}m")
print(f"- debt + cash                : {implied_equity:8,.0f}m  (implied equity value)")
print(f"/ shares outstanding         : {target['shares_m']:8,.0f}m")
print(f"= IMPLIED SHARE PRICE        : ${implied_price:7.2f}")
print(f"  actual market price        : ${target['price_usd']:7.2f}")
print()
print(f"In plain words: at a TYPICAL peer rating, Apple would be worth about ${implied_price:.0f}.")
print(f"The market pays ${target['price_usd']:.0f} - roughly {target['price_usd']/implied_price-1:.0%} more. Is that premium justified?")
print("One number cannot say. The next exercise asks all seven ratings instead of only the median.")

The seven peer ratings (EV/EBITDA):
ticker
MSFT    19.0
NVDA    39.0
GOOGL   28.4
AMZN    19.5
META    14.6
CRM     21.2
ORCL    18.4 

peer median EV/EBITDA        :     19.5x
x Apple's EBITDA             :  144,748m
= implied enterprise value   : 2,825,256m
- debt + cash                : 2,780,456m  (implied equity value)
/ shares outstanding         :   14,594m
= IMPLIED SHARE PRICE        : $ 190.52
  actual market price        : $ 309.35

In plain words: at a TYPICAL peer rating, Apple would be worth about $191.
The market pays $309 - roughly 62% more. Is that premium justified?
One number cannot say. The next exercise asks all seven ratings instead of only the median.


### Exercise 4: from a point to a range

The deterministic answer used one rating, the median. But no law says Apple deserves the *median* rating — so the honest next step is to ask **all seven ratings the market actually prints**: apply each one through the same chain and read the whole set of implied prices. Seven inputs, seven answers, no sampling needed.

One caution when reading the results: every implied price in this exercise is **Apple's** — "Apple at NVIDIA's rating" means *Apple's EBITDA priced at NVIDIA's multiple*, which has nothing to do with NVIDIA's own share price.

The gaps are step 2 again: the inverted bridge.

In [11]:
def implied_prices(df: pd.DataFrame, ticker: str) -> pd.Series:
    """Apply EVERY peer rating to one company; return implied prices indexed by peer."""
    row = df.set_index("ticker").loc[ticker]
    ratings = df.loc[df["ticker"] != ticker].set_index("ticker")["ev_ebitda"].dropna()
### START CODE HERE ###
    implied_ev = ratings * row["ebitda_m"]                    # each rating prices which figure?
    implied_mcap = implied_ev - row["total_debt_m"] + row["cash_m"]   # invert the EV bridge
    implied_price = implied_mcap / row["shares_m"]
### END CODE HERE ###
    return implied_price

In [12]:
# ✅ self-check: run me (offline, deterministic). One peer rating of exactly 10x,
# EBITDA 100, debt 40, cash 20, 10 shares -> implied price must be exactly 98.
toy = pd.DataFrame({"ticker": ["TARGET", "PEER"], "ebitda_m": [100.0, 50.0],
                    "total_debt_m": [40.0, 0.0], "cash_m": [20.0, 0.0],
                    "shares_m": [10.0, 1.0], "ev_ebitda": [float("nan"), 10.0]})
try:
    result = implied_prices(toy, "TARGET")
except KeyError as e:
    raise AssertionError(
        f"your bridge references column {e}, which the bridge does not use. "
        "The EV bridge inverts with DEBT and CASH: implied equity = implied EV "
        "- total_debt_m + cash_m. (D&A belongs to the EBITDA formula, not the bridge.)"
    ) from None
assert abs(result.iloc[0] - 98.0) < 1e-9, "check the EV bridge inversion: (10*100 - 40 + 20) / 10 = 98"
print("All checks passed ✅  The bridge inverts correctly. Now Apple, at every peer's rating:")

All checks passed ✅  The bridge inverts correctly. Now Apple, at every peer's rating:


In [13]:
apple = implied_prices(companies, "AAPL").sort_values()
actual = companies.set_index("ticker").loc["AAPL", "price_usd"]
print("What Apple would be worth at each peer's rating:")
for peer, price in apple.items():
    marker = "  <- above the actual price" if price > actual else ""
    print(f"  at {peer:5}'s rating: ${price:7,.0f}{marker}")
print(f"\nRange: ${apple.min():,.0f} to ${apple.max():,.0f} | median ${apple.median():,.0f} | actual ${actual:,.2f}")

# The same tool, applied to every company:
rows = []
for t in companies["ticker"]:
    ip = implied_prices(companies, t)
    rows.append({"ticker": t, "low": ip.min(), "median": ip.median(), "high": ip.max(),
                 "actual": companies.set_index("ticker").loc[t, "price_usd"]})
ranges = pd.DataFrame(rows)

import importlib, subprocess
for _pkg in ("plotly", "nbformat"):      # self-heal, in case this cell runs on a fresh kernel
    try:
        importlib.import_module(_pkg)
    except ModuleNotFoundError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", _pkg], check=True)
import plotly.graph_objects as go
fig = go.Figure()
fig.add_bar(x=ranges["ticker"], y=ranges["high"] - ranges["low"], base=ranges["low"],
            name="peer-implied range (lowest to highest rating)", marker_color="lightsteelblue",
            hovertemplate="low %{base:$,.0f} - high %{y:$,.0f}<extra></extra>")
fig.add_scatter(x=ranges["ticker"], y=ranges["actual"], mode="markers", name="actual price",
                marker=dict(color="firebrick", size=10, symbol="diamond"))
fig.add_scatter(x=ranges["ticker"], y=ranges["median"], mode="markers", name="median implied",
                marker=dict(color="steelblue", size=8))
fig.update_layout(title="What peer ratings imply each share is worth",
                  yaxis_title="share price (USD)")
fig.show()

What Apple would be worth at each peer's rating:
  at META 's rating: $    141
  at ORCL 's rating: $    180
  at MSFT 's rating: $    186
  at AMZN 's rating: $    191
  at CRM  's rating: $    207
  at GOOGL's rating: $    278
  at NVDA 's rating: $    383  <- above the actual price

Range: $141 to $383 | median $191 | actual $309.35


*Reading the results, plainly.* Each implied price above is **Apple's**: "at NVIDIA's rating" means Apple's EBITDA priced at NVIDIA's multiple — it says nothing about NVIDIA's own share price. The chart then answers the same question for every company: the bar spans the implied prices at the lowest and highest peer rating, the blue dot is the median, the red diamond the actual market price.

**Apple's row, spelled out — this is the session's finding:**

1. At a *typical* peer rating (the median), Apple would be worth about **$191**. The market pays **$309** — a premium of roughly **60%**.
2. Could *any* peer rating justify $309? Only one: **NVIDIA's** — the highest in the group — implies about **$383** for Apple. Every other peer's rating implies less. The market therefore prices Apple **above what six of its seven peers' ratings would justify**.
3. The honest conclusion, in one sentence: **the market prices Apple as if it were nearly the best company in this peer group.** Not "Apple is overpriced" — the method cannot say that — but "Apple is priced for near-best-in-group performance."
4. What comparable-company analysis *cannot* tell you is whether that confidence is deserved: whether Apple's growth and quality genuinely merit an almost-NVIDIA rating. **That judgment needs evidence about growth — and Session 3's earnings engine is built to gather exactly that.**

**What the range bought you:** the single median invited a crude verdict ("60% overpriced"); the seven ratings turned it into a precise, defensible statement ("priced above six of seven peers' ratings"). A range with a stated assumption is an analysis; a point without one is an opinion.

## Part C · Publish to GitHub
Your comparable-company analysis tool is a project. In the **terminal** (not this notebook):

```bash
git init && git add . && git commit -m "Comps tool: first working version, AAPL multiple hand-verified"
gh repo create my-finance-toolkit --private --source . --push
```

(No `gh`? github.com → New repo → follow "push an existing repository". Cheatsheet: `cheatsheets/git-github-for-finance.md`.)

## Deliverable checklist

- [ ] All three ✅ checks green, with at least one exercise implemented via Claude Code (`Option/Alt+K` on the docstring)
- [ ] `outputs/comps_summary.csv` exists
- [ ] Repo pushed to GitHub with ≥2 commits
- [ ] Stretch: scatter `revenue_growth_1y` vs `ev_ebitda`: is growth priced in?

**Next:** `03-debugging-analytics.ipynb`: your Apple valuation, broken on purpose.